# 🏎️ Predicting Formula 1 Podium Finishes Using Supervised Machine Learning
**DS120B — Machine Learning | Summer Semester 2026**  
**Dataset:** Formula 1 Races Data from 2000–2025  
**Source:** https://www.kaggle.com/datasets/syedzubairahmed0022/formula-1-races-data-from-2000-2025  
**Task:** Binary Classification — Predicting whether a driver finishes on the podium (Top 3)

## 0. Setup & Library Imports

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── XGBoost ──────────────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    print('XGBoost not found. Installing...')
    import subprocess; subprocess.run(['pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True

# ── imbalanced-learn (SMOTE) ──────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    print('imbalanced-learn not found. Installing...')
    import subprocess; subprocess.run(['pip', 'install', 'imbalanced-learn', '-q'])
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True

# ── Plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#1F3864', '#2E75B6', '#5B9BD5', '#A9C4E4', '#D6E4F0']
F1_RED = '#E10600'
F1_DARK = '#1F3864'
print('✅ All libraries loaded successfully.')

## 1. Data Loading

> **Download instructions:** Obtain `formula1_races.csv` from [Kaggle](https://www.kaggle.com/datasets/syedzubairahmed0022/formula-1-races-data-from-2000-2025) and place it in the same directory as this notebook.

In [ ]:
import os, glob

# ── Locate the CSV (handles various filenames) ────────────────────────────────
candidates = glob.glob('*.csv') + glob.glob('**/*.csv', recursive=True)
csv_file = None
for c in candidates:
    if 'formula' in c.lower() or 'f1' in c.lower() or 'race' in c.lower():
        csv_file = c; break
if csv_file is None and candidates:
    csv_file = candidates[0]   # fall back to first CSV found

if csv_file:
    df_raw = pd.read_csv(csv_file)
    print(f'Loaded: {csv_file}')
else:
    # ── Synthetic fallback so the notebook runs end-to-end ────────────────────
    print('⚠️  Dataset not found locally. Generating synthetic data for demonstration.')
    np.random.seed(42)
    n = 10398
    constructors = ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren', 'Alpine',
                    'Aston Martin', 'Williams', 'AlphaTauri', 'Alfa Romeo', 'Haas']
    circuits = ['Monaco', 'Silverstone', 'Monza', 'Spa', 'Suzuka',
                'Bahrain', 'Melbourne', 'Singapore', 'Interlagos', 'Hungaroring']
    nationalities = ['British', 'German', 'Dutch', 'Spanish', 'French',
                     'Italian', 'Brazilian', 'Australian', 'Finnish', 'Mexican']
    grid = np.random.randint(1, 21, n)
    # Simulate realistic position: lower grid → better position
    noise = np.random.normal(0, 4, n)
    pos_raw = np.clip(grid + noise, 1, 20)
    position = np.round(pos_raw).astype(int)
    position = np.clip(position, 1, 20)

    df_raw = pd.DataFrame({
        'year': np.random.randint(2000, 2026, n),
        'round': np.random.randint(1, 24, n),
        'circuit_name': np.random.choice(circuits, n),
        'driver_name': [f'Driver_{i%25+1}' for i in range(n)],
        'driver_nationality': np.random.choice(nationalities, n),
        'constructor_name': np.random.choice(constructors, n),
        'grid': grid,
        'position': position,
        'points': np.where(position == 1, 25, np.where(position == 2, 18,
                   np.where(position == 3, 15, np.where(position <= 10,
                   np.random.randint(1, 12, n), 0)))),
        'laps': np.random.randint(40, 78, n),
        'status': np.random.choice(['Finished', 'Finished', 'Finished', 'Retired', 'Accident'], n),
        'fastest_lap_rank': np.random.randint(1, 21, n),
        'fastest_lap_speed': np.random.uniform(180, 260, n),
        'q1_time': np.random.uniform(80, 100, n),
        'q2_time': np.random.uniform(79, 99, n),
        'q3_time': np.random.uniform(78, 98, n),
        'driver_wins': np.random.randint(0, 100, n),
        'driver_podiums': np.random.randint(0, 200, n),
        'constructor_wins': np.random.randint(0, 500, n),
        'weather': np.random.choice(['Dry', 'Wet', 'Mixed'], n, p=[0.75, 0.15, 0.10]),
    })

print(f'Shape: {df_raw.shape}')
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===');
df_raw.info()
print('\n=== Descriptive Statistics ===');
df_raw.describe()

In [ ]:
print('=== Missing Values ===');
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})[missing > 0]

In [ ]:
# ── Determine position column ─────────────────────────────────────────────────
pos_col = None
for c in ['position', 'positionOrder', 'Position', 'finish_position', 'finishing_position', 'pos']:
    if c in df_raw.columns:
        pos_col = c; break
if pos_col is None:
    # Try to find any column with 'pos' in name
    cols_with_pos = [c for c in df_raw.columns if 'pos' in c.lower()]
    pos_col = cols_with_pos[0] if cols_with_pos else df_raw.columns[0]
print(f'Using position column: "{pos_col}"')

# ── Engineer target variable ──────────────────────────────────────────────────
df_raw[pos_col] = pd.to_numeric(df_raw[pos_col], errors='coerce')
df = df_raw.dropna(subset=[pos_col]).copy()
df['podium'] = (df[pos_col] <= 3).astype(int)

print(f'\nTarget distribution:')
vc = df['podium'].value_counts()
print(f'  Non-podium (0): {vc.get(0, 0):,} ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'  Podium     (1): {vc.get(1, 0):,} ({vc.get(1,0)/len(df)*100:.1f}%)')

In [ ]:
# ── Figure 1: Target Class Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 1. Target Variable Distribution — Podium vs. Non-Podium', fontsize=13, fontweight='bold')

counts = df['podium'].value_counts().sort_index()
labels = ['Non-Podium (0)', 'Podium (1)']
colors = [F1_DARK, F1_RED]

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Counts', fontweight='bold')
axes[0].set_ylabel('Number of Races')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig('fig1_class_distribution.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Figure 2: Grid Position vs Podium Rate ────────────────────────────────────
grid_col = None
for c in ['grid', 'Grid', 'grid_position', 'gridPosition', 'starting_grid']:
    if c in df.columns:
        grid_col = c; break

if grid_col:
    df[grid_col] = pd.to_numeric(df[grid_col], errors='coerce')
    grid_podium = df.groupby(grid_col)['podium'].mean().reset_index()
    grid_podium = grid_podium[grid_podium[grid_col].between(1, 20)]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(grid_podium[grid_col], grid_podium['podium'] * 100,
           color=[F1_RED if x <= 3 else F1_DARK for x in grid_podium[grid_col]],
           edgecolor='white', linewidth=0.8)
    ax.set_xlabel('Starting Grid Position', fontsize=11)
    ax.set_ylabel('Podium Finish Rate (%)', fontsize=11)
    ax.set_title('Figure 2. Grid Position vs. Podium Finish Rate', fontsize=13, fontweight='bold')
    ax.set_xticks(range(1, 21))
    ax.axhline(y=15, color='gray', linestyle='--', alpha=0.6, label='Overall podium rate (~15%)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('fig2_grid_vs_podium.png', bbox_inches='tight', dpi=120)
    plt.show()
else:
    print('Grid column not found — skipping Figure 2.')

In [ ]:
# ── Figure 3: Correlation Heatmap ─────────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude the raw position column to avoid leakage
num_cols = [c for c in num_cols if c != pos_col]

if len(num_cols) >= 3:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(max(8, len(num_cols)), max(6, len(num_cols)-2)))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
    ax.set_title('Figure 3. Correlation Heatmap of Numerical Features', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig3_correlation_heatmap.png', bbox_inches='tight', dpi=120)
    plt.show()

## 3. Data Preprocessing

**RQ3:** Investigating the effect of different preprocessing strategies.

In [ ]:
# ── Identify feature columns ──────────────────────────────────────────────────
DROP_COLS = [pos_col, 'podium',
             'time', 'Time', 'race_time',        # post-race leakage
             'points', 'Points',                  # awarded after race = leakage
             'driver_name', 'Driver', 'driver',   # too high cardinality
             'date', 'Date', 'raceId', 'race_id']

feature_cols = [c for c in df.columns if c not in DROP_COLS]
print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

X = df[feature_cols].copy()
y = df['podium'].copy()

# ── Separate numeric and categorical ─────────────────────────────────────────
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols_feat = X.select_dtypes(include=[np.number]).columns.tolist()

print(f'\nNumerical features ({len(num_cols_feat)}): {num_cols_feat}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

# ── Encode categorical features ───────────────────────────────────────────────
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = X[c].fillna('Unknown')
    X[c] = le.fit_transform(X[c].astype(str))
    le_dict[c] = le

# ── Impute missing numerics ───────────────────────────────────────────────────
imp = SimpleImputer(strategy='median')
X[num_cols_feat] = imp.fit_transform(X[num_cols_feat])

print('\n✅ Preprocessing complete.')
print(f'X shape: {X.shape}, y distribution: {y.value_counts().to_dict()}')

In [ ]:
# ── Train / Test Split (stratified, 80/20) ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Feature Scaling ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

# ── SMOTE for class imbalance ─────────────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print(f'Train size: {len(X_train):,} | Test size: {len(X_test):,}')
print(f'After SMOTE — train size: {len(X_train_sm):,}')
print(f'Train class distribution after SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}')

## 4. Model Training & Evaluation

### RQ1 — Baseline Performance

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, fit_first=True):
    if fit_first:
        model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
    metrics = {
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_te, y_pred, zero_division=0), 4),
        'F1-Score':  round(f1_score(y_te, y_pred, zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_te, y_prob), 4) if y_prob is not None else 'N/A',
    }
    return metrics, y_pred, y_prob

# ── Baseline models (no SMOTE, scaled features) ───────────────────────────────
baselines = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
    'k-NN':               KNeighborsClassifier(n_neighbors=7),
}

baseline_results = {}
for name, model in baselines.items():
    metrics, _, _ = evaluate_model(name, model, X_train_sc, y_train, X_test_sc, y_test)
    baseline_results[name] = metrics
    print(f'{name:25s} | Acc: {metrics["Accuracy"]:.4f} | F1: {metrics["F1-Score"]:.4f} | AUC: {metrics["AUC-ROC"]}')

df_baseline = pd.DataFrame(baseline_results).T
print('\n=== Table I. Baseline Model Performance ===')
print(df_baseline.to_string())

In [ ]:
# ── Figure 4: Baseline Performance Comparison ─────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_to_plot))
width = 0.22
colors_b = [F1_DARK, '#2E75B6', '#5B9BD5']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, row) in enumerate(df_baseline.iterrows()):
    vals = [float(row[m]) for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=name, color=colors_b[i], alpha=0.9)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Figure 4. Baseline Model Performance on F1 Podium Dataset', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plt.savefig('fig4_baseline_performance.png', bbox_inches='tight', dpi=120)
plt.show()

### RQ2 — Full Model Comparison

In [ ]:
# ── All five candidate models (trained on SMOTE data) ─────────────────────────
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pw = neg / pos  # for XGBoost

all_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
                                          scale_pos_weight=scale_pw, random_state=42,
                                          eval_metric='logloss', verbosity=0),
    'SVM':                 SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True, random_state=42),
}

all_results = {}
trained_models = {}
y_probs_all = {}

for name, model in all_models.items():
    print(f'Training {name}...', end=' ')
    metrics, y_pred, y_prob = evaluate_model(name, model, X_train_sm, y_train_sm, X_test_sc, y_test)
    all_results[name] = metrics
    trained_models[name] = model
    y_probs_all[name] = y_prob
    print(f'F1={metrics["F1-Score"]:.4f}  AUC={metrics["AUC-ROC"]}')

df_all = pd.DataFrame(all_results).T
print('\n=== Table II. Comparative Model Performance ===')
print(df_all.to_string())

In [ ]:
# ── Figure 5: Full Model Comparison ──────────────────────────────────────────
metrics_comp = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
models_list = list(df_all.index)
x2 = np.arange(len(metrics_comp))
w2 = 0.14
colors_all = ['#1F3864', '#2E75B6', '#5B9BD5', F1_RED, '#A9C4E4']

fig, ax = plt.subplots(figsize=(14, 5))
for i, name in enumerate(models_list):
    vals = [float(df_all.loc[name, m]) if df_all.loc[name, m] != 'N/A' else 0 for m in metrics_comp]
    ax.bar(x2 + i * w2, vals, w2, label=name, color=colors_all[i], alpha=0.88)

ax.set_xticks(x2 + w2 * 2)
ax.set_xticklabels(metrics_comp, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Figure 5. Comparative Performance of All Candidate Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('fig5_model_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Figure 8: AUC-ROC Curves ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#1F3864', '#2E75B6', '#5B9BD5', F1_RED, '#A9C4E4']

for (name, y_prob), color in zip(y_probs_all.items(), colors_roc):
    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('Figure 8. AUC-ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig8_roc_curves.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Figure 9: Confusion Matrix (best model) ───────────────────────────────────
best_model_name = df_all['F1-Score'].astype(float).idxmax()
print(f'Best model by F1-Score: {best_model_name}')

best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Podium', 'Podium'],
            yticklabels=['Non-Podium', 'Podium'],
            linewidths=1, linecolor='white')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Figure 9. Confusion Matrix — {best_model_name}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_confusion_matrix.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n=== Classification Report ===')
print(classification_report(y_test, y_pred_best, target_names=['Non-Podium', 'Podium']))

### RQ3 — Effect of Preprocessing (Ablation Study)

In [ ]:
# Use Random Forest for the ablation (robust model, consistent results)
rf_abl = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

ablation_results = {}

# Strategy 1: Raw data (no scaling, no SMOTE)
rf_abl.fit(X_train, y_train)
m, _, _ = evaluate_model('Raw data', rf_abl, X_train, y_train, X_test, y_test, fit_first=False)
ablation_results['Raw data'] = m

# Strategy 2: Imputation only (already done)
m, _, _ = evaluate_model('Imputation only', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                         X_train, y_train, X_test, y_test)
ablation_results['Imputation only'] = m

# Strategy 3: Scaling + encoding
m, _, _ = evaluate_model('Scaling + Encoding', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                         X_train_sc, y_train, X_test_sc, y_test)
ablation_results['Scaling + Encoding'] = m

# Strategy 4: Full pipeline (Scaling + Encoding + SMOTE)
m, _, _ = evaluate_model('Full Pipeline (+ SMOTE)', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                         X_train_sm, y_train_sm, X_test_sc, y_test)
ablation_results['Full Pipeline (+ SMOTE)'] = m

df_abl = pd.DataFrame(ablation_results).T
print('=== Table III. Impact of Preprocessing Strategies ===')
print(df_abl.to_string())

In [ ]:
# ── Figure 6: Preprocessing Ablation ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
metrics_abl = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
strategies = list(df_abl.index)
x_abl = np.arange(len(strategies))

for j, m in enumerate(metrics_abl):
    vals = df_abl[m].astype(float).values
    ax.plot(x_abl, vals, marker='o', linewidth=2, label=m, color=colors_all[j])

ax.set_xticks(x_abl)
ax.set_xticklabels(strategies, fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0.4, 1.0)
ax.set_title('Figure 6. Effect of Preprocessing Strategies on Random Forest Performance', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig6_preprocessing_ablation.png', bbox_inches='tight', dpi=120)
plt.show()

### RQ4 — Feature Importance & Interpretability

In [ ]:
# Use the best tree-based model for feature importance
tree_based = ['Random Forest', 'XGBoost', 'Decision Tree']
fi_model_name = next((m for m in tree_based if m in trained_models), None)

if fi_model_name:
    fi_model = trained_models[fi_model_name]
    importances = fi_model.feature_importances_
    fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
    fi_df = fi_df.sort_values('Importance', ascending=False).head(15)

    print(f'=== Table IV. Top-15 Features — {fi_model_name} ===')
    fi_df['Rank'] = range(1, len(fi_df)+1)
    print(fi_df[['Rank', 'Feature', 'Importance']].to_string(index=False))

    # ── Figure 7: Feature Importance ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 6))
    colors_fi = [F1_RED if i < 3 else F1_DARK for i in range(len(fi_df))]
    ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1], color=colors_fi[::-1])
    ax.set_xlabel('Feature Importance Score', fontsize=11)
    ax.set_title(f'Figure 7. Top-15 Feature Importances — {fi_model_name}', fontsize=12, fontweight='bold')
    ax.axvline(x=fi_df['Importance'].mean(), color='gray', linestyle='--', alpha=0.6, label='Mean importance')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('fig7_feature_importance.png', bbox_inches='tight', dpi=120)
    plt.show()

### RQ5 — Model Ranking Across Metrics

In [ ]:
# ── Table V & Figure 11: Rankings across metrics ──────────────────────────────
rank_metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
rank_df = pd.DataFrame(index=df_all.index)

for m in rank_metrics:
    vals = df_all[m].apply(lambda x: float(x) if x != 'N/A' else 0)
    rank_df[f'Rank by {m}'] = vals.rank(ascending=False).astype(int)

print('=== Table V. Model Rankings Across Evaluation Metrics ===')
print(rank_df.to_string())

# ── Figure 11: Bump / Slope Chart ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x_bump = np.arange(len(rank_metrics))

for i, model_name in enumerate(rank_df.index):
    ranks = rank_df.loc[model_name].values
    ax.plot(x_bump, ranks, marker='o', linewidth=2.5, label=model_name, color=colors_all[i])
    ax.text(x_bump[-1] + 0.05, ranks[-1], model_name, fontsize=8.5, va='center', color=colors_all[i])

ax.set_xticks(x_bump)
ax.set_xticklabels(rank_metrics, fontsize=10)
ax.set_ylabel('Rank (1 = Best)', fontsize=11)
ax.set_ylim(0.5, len(all_models) + 0.5)
ax.invert_yaxis()
ax.set_title('Figure 11. Model Rankings Across Evaluation Metrics (Bump Chart)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, axis='y')
plt.tight_layout()
plt.savefig('fig11_bump_chart.png', bbox_inches='tight', dpi=120)
plt.show()

### RQ6 — Robustness and Generalization

In [ ]:
# ── Stratified K-Fold Cross-Validation ────────────────────────────────────────
print('Running 5-fold and 10-fold cross-validation on best model...')

best_for_cv = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
skf_5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_scoring = ['accuracy', 'f1', 'roc_auc']
cv5  = cross_validate(best_for_cv, X_train_sc, y_train, cv=skf_5,  scoring=cv_scoring, n_jobs=-1)
cv10 = cross_validate(best_for_cv, X_train_sc, y_train, cv=skf_10, scoring=cv_scoring, n_jobs=-1)

print('\n=== Table VI. Cross-Validation Results — Random Forest ===')
print(f'{"Setting":<20} | {"Accuracy":>10} | {"F1-Score":>10} | {"AUC-ROC":>10}')
print('-' * 60)
for label, cv in [('5-fold CV', cv5), ('10-fold CV', cv10)]:
    acc = f"{cv['test_accuracy'].mean():.4f} ±{cv['test_accuracy'].std():.4f}"
    f1  = f"{cv['test_f1'].mean():.4f} ±{cv['test_f1'].std():.4f}"
    auc = f"{cv['test_roc_auc'].mean():.4f} ±{cv['test_roc_auc'].std():.4f}"
    print(f'{label:<20} | {acc:>10} | {f1:>10} | {auc:>10}')

In [ ]:
# ── Robustness: Noise Injection ───────────────────────────────────────────────
print('Testing robustness under noise injection...')
noise_results = {}

for noise_pct in [0, 5, 10, 20]:
    X_noisy = X_train_sc.copy()
    if noise_pct > 0:
        mask = np.random.choice([True, False], size=X_noisy.shape,
                                p=[noise_pct/100, 1 - noise_pct/100])
        X_noisy = X_noisy.mask(mask, other=np.random.randn(*X_noisy.shape))
    rf_n = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_n.fit(X_noisy, y_train)
    y_pn = rf_n.predict(X_test_sc)
    noise_results[f'{noise_pct}% noise'] = {
        'F1-Score': round(f1_score(y_test, y_pn, zero_division=0), 4),
        'AUC-ROC':  round(roc_auc_score(y_test, rf_n.predict_proba(X_test_sc)[:, 1]), 4)
    }

print('\n=== Noise Robustness Results ===')
for scenario, m in noise_results.items():
    print(f'{scenario:15s} | F1={m["F1-Score"]:.4f} | AUC={m["AUC-ROC"]:.4f}')

In [ ]:
# ── Figure 10: Robustness Boxplot from CV ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Figure 10. Model Robustness Under Cross-Validation', fontsize=13, fontweight='bold')

axes[0].boxplot([cv5['test_f1'], cv10['test_f1']],
                labels=['5-Fold CV', '10-Fold CV'],
                patch_artist=True,
                boxprops=dict(facecolor='#D6E4F0'),
                medianprops=dict(color=F1_RED, linewidth=2))
axes[0].set_ylabel('F1-Score')
axes[0].set_title('F1-Score Distribution')

axes[1].boxplot([cv5['test_roc_auc'], cv10['test_roc_auc']],
                labels=['5-Fold CV', '10-Fold CV'],
                patch_artist=True,
                boxprops=dict(facecolor='#D6E4F0'),
                medianprops=dict(color=F1_RED, linewidth=2))
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC Distribution')

plt.tight_layout()
plt.savefig('fig10_robustness_boxplot.png', bbox_inches='tight', dpi=120)
plt.show()

### RQ7 — Practical Usefulness & Final Recommendation

In [ ]:
# ── Figure 12: Radar Chart — Final Trade-off ──────────────────────────────────
from matplotlib.patches import FancyArrowPatch
from matplotlib.path import Path

criteria = ['Predictive\nPerformance', 'Interpretability', 'Robustness', 'Speed', 'Deployment']
# Scores 1–5 for each model on each criterion
model_scores = {
    'Logistic Regression': [3, 5, 3, 5, 5],
    'Decision Tree':        [3, 5, 3, 5, 4],
    'Random Forest':        [4, 3, 5, 3, 4],
    'XGBoost':              [5, 2, 4, 2, 3],
    'SVM':                  [3, 2, 3, 1, 3],
}

N = len(criteria)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
for i, (name, scores) in enumerate(model_scores.items()):
    vals = scores + scores[:1]
    ax.plot(angles, vals, color=colors_all[i], linewidth=2, label=name)
    ax.fill(angles, vals, color=colors_all[i], alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=10)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=7)
ax.set_title('Figure 12. Final Model Trade-off Radar Chart', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig('fig12_radar_chart.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# ── Table VII. Final Model Decision Matrix ────────────────────────────────────
decision_matrix = pd.DataFrame({
    'Criterion':             ['Predictive Performance', 'Interpretability', 'Robustness', 'Computational Cost', 'Deployment Suitability'],
    'Logistic Regression':   ['Medium', 'High', 'Medium', 'Low', 'High'],
    'Decision Tree':         ['Medium', 'High', 'Medium', 'Low', 'High'],
    'Random Forest':         ['High', 'Medium', 'High', 'Medium', 'High'],
    'XGBoost':               ['Very High', 'Low-Medium', 'High', 'High', 'Medium'],
    'SVM':                   ['Medium-High', 'Low', 'Medium', 'High', 'Medium'],
})
print('=== Table VII. Final Model Decision Matrix ===')
print(decision_matrix.to_string(index=False))

print('\n' + '='*70)
print('FINAL RECOMMENDATION')
print('='*70)
print(f'Best F1-Score model: {df_all["F1-Score"].astype(float).idxmax()}')
print(f'Best AUC-ROC model:  {df_all["AUC-ROC"].apply(lambda x: float(x) if x != "N/A" else 0).idxmax()}')
print('\nFor production deployment, Random Forest offers the best balance')
print('of performance, robustness, interpretability, and deployment ease.')
print('XGBoost is recommended when maximum predictive performance is required.')

## 5. Summary of All Results

| | Acc | Prec | Recall | F1 | AUC |
|--|--|--|--|--|--|
| See `df_all` table above for full results |

In [ ]:
print('=== COMPLETE RESULTS SUMMARY ===')
print('\n[Table I] Baseline Models:')
print(df_baseline.to_string())
print('\n[Table II] All Models (with SMOTE):')
print(df_all.to_string())
print('\n[Table III] Preprocessing Ablation:')
print(df_abl.to_string())
print('\n[Table V] Model Rankings:')
print(rank_df.to_string())
print('\n[Table VI] Noise Robustness:')
print(pd.DataFrame(noise_results).T.to_string())
print('\n✅ All analyses complete. Figures saved as PNG files.')